# LumenY 6.0 — 01: MFE Labels & Feature Augmentation

**Goal:** Build the dataset for a big-move detection model.

**Inputs:**
- Existing 64 microstructure features from `features_2/` (7 majors) and `features_3/` (8 crosses)
- 1H OHLCV from `processed/` (for label computation + extra features)

**Outputs:** `features_6/` — one parquet per pair with:
- Original 64 microstructure features (unchanged)
- MFE-based labels (trailing stop simulation, no fixed horizon)
- ~20 additional contextual features (ATR, session, range position, feature deltas)

**Key concept:** The label simulates a trailing-stop trade with no time limit.
For each bar, we walk forward and measure how far price moves in one direction
before a trailing stop is hit. This is the Maximum Favorable Excursion (MFE).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──
PROCESSED_DIR = Path('../backend/data/processed')
FEATURES_2_DIR = Path('../backend/data/features_2')  # 7 major pairs
FEATURES_3_DIR = Path('../backend/data/features_3')  # 8 cross pairs
OUTPUT_DIR = Path('../backend/data/features_6')
OUTPUT_DIR.mkdir(exist_ok=True)

PAIRS_MAJORS = ['EURUSD', 'GBPUSD', 'AUDUSD', 'NZDUSD', 'USDCAD', 'USDCHF', 'USDJPY']
PAIRS_CROSSES = ['EURJPY', 'GBPJPY', 'EURGBP', 'EURAUD', 'AUDJPY', 'CADJPY', 'CHFJPY', 'AUDNZD']
ALL_PAIRS = PAIRS_MAJORS + PAIRS_CROSSES

# Pip sizes
PIP_SIZE = {
    'EURUSD': 0.0001, 'GBPUSD': 0.0001, 'AUDUSD': 0.0001, 'NZDUSD': 0.0001,
    'USDCAD': 0.0001, 'USDCHF': 0.0001, 'USDJPY': 0.01,
    'EURJPY': 0.01, 'GBPJPY': 0.01, 'EURGBP': 0.0001, 'EURAUD': 0.0001,
    'AUDJPY': 0.01, 'CADJPY': 0.01, 'CHFJPY': 0.01, 'AUDNZD': 0.0001,
}

TRAIN_END = '2024-06-30'

print(f'Pairs: {len(ALL_PAIRS)}')
print(f'Output: {OUTPUT_DIR}')

## 1. Load Existing Microstructure Features

In [ ]:
def load_microstructure_features(pair):
    """Load pre-computed microstructure features for a pair."""
    if pair in PAIRS_MAJORS:
        path = FEATURES_2_DIR / f'{pair}_microstructure.parquet'
    else:
        path = FEATURES_3_DIR / f'{pair}_microstructure.parquet'
    
    df = pd.read_parquet(path)
    if 'datetime' in df.columns:
        df = df.set_index('datetime')
    df.index = pd.to_datetime(df.index)
    return df

# Quick check
df_test = load_microstructure_features('EURUSD')
print(f'EURUSD features: {df_test.shape}')
print(f'Date range: {df_test.index.min().date()} to {df_test.index.max().date()}')
print(f'Columns: {len([c for c in df_test.columns if c != "pair"])} features + pair')
del df_test

## 2. MFE Label Computation

For each hourly bar, we simulate two trailing-stop trades (long and short):

**Long trade at bar t:**
1. Enter at `close[t]`
2. Walk forward bar by bar, tracking `running_max` of highs
3. At each bar, if `running_max - low[t+k]` exceeds the trailing stop → trade ends
4. MFE_long = `running_max - entry` (in pips)

**Short trade at bar t:**
1. Enter at `close[t]`  
2. Walk forward bar by bar, tracking `running_min` of lows
3. At each bar, if `high[t+k] - running_min` exceeds the trailing stop → trade ends
4. MFE_short = `entry - running_min` (in pips)

The trailing stop is ATR-based per pair to normalize across different volatilities.
We cap the forward walk at 72 hours (3 days) as a safety limit.

In [ ]:
def compute_mfe_labels(df_1h, pair, trail_atr_mult=1.5, max_bars=72):
    """Compute MFE labels for long and short trailing-stop trades.
    
    Args:
        df_1h: 1H OHLCV DataFrame (datetime index)
        pair: pair name for pip conversion
        trail_atr_mult: trailing stop = trail_atr_mult * ATR_24
        max_bars: maximum forward walk (72h = 3 days)
    
    Returns:
        DataFrame with MFE columns added
    """
    pip = PIP_SIZE[pair]
    highs = df_1h['high'].values
    lows = df_1h['low'].values
    closes = df_1h['close'].values
    n = len(df_1h)
    
    # ATR-24 for adaptive trailing stop
    tr = np.maximum(
        highs - lows,
        np.maximum(
            np.abs(highs - np.roll(closes, 1)),
            np.abs(lows - np.roll(closes, 1))
        )
    )
    tr[0] = highs[0] - lows[0]  # fix first bar
    atr_24 = pd.Series(tr).rolling(24, min_periods=6).mean().values
    
    # Output arrays
    mfe_long_pips = np.full(n, np.nan)
    mfe_short_pips = np.full(n, np.nan)
    trail_long_bars = np.full(n, np.nan)   # how many bars the long trade lasted
    trail_short_bars = np.full(n, np.nan)
    trail_stop_pips = np.full(n, np.nan)   # the trailing stop used (for reference)
    
    for i in range(n - 1):
        if np.isnan(atr_24[i]) or atr_24[i] < 1e-10:
            continue
        
        entry = closes[i]
        stop_dist = trail_atr_mult * atr_24[i]  # trailing stop distance in price
        trail_stop_pips[i] = stop_dist / pip
        
        # ── Long trade ──
        running_max = entry
        last_k = i  # track last bar index
        for k in range(i + 1, min(i + 1 + max_bars, n)):
            running_max = max(running_max, highs[k])
            drawdown = running_max - lows[k]
            last_k = k
            if drawdown >= stop_dist:
                break
        mfe_long_pips[i] = (running_max - entry) / pip
        trail_long_bars[i] = last_k - i
        
        # ── Short trade ──
        running_min = entry
        last_k = i
        for k in range(i + 1, min(i + 1 + max_bars, n)):
            running_min = min(running_min, lows[k])
            runup = highs[k] - running_min
            last_k = k
            if runup >= stop_dist:
                break
        mfe_short_pips[i] = (entry - running_min) / pip
        trail_short_bars[i] = last_k - i
    
    return pd.DataFrame({
        'mfe_long_pips': mfe_long_pips,
        'mfe_short_pips': mfe_short_pips,
        'trail_long_bars': trail_long_bars,
        'trail_short_bars': trail_short_bars,
        'trail_stop_pips': trail_stop_pips,
        'mfe_atr_24': atr_24 / pip,  # in pips for readability
    }, index=df_1h.index)

print('MFE label function defined.')

In [ ]:
# Test on one pair
df_1h = pd.read_parquet(PROCESSED_DIR / 'EURUSD_1H.parquet')
if 'datetime' in df_1h.columns:
    df_1h = df_1h.set_index('datetime')
df_1h.index = pd.to_datetime(df_1h.index)

df_labels = compute_mfe_labels(df_1h, 'EURUSD')

print(f'EURUSD MFE labels: {df_labels.shape}')
print(f'\nMFE Long (pips):')
print(df_labels['mfe_long_pips'].describe())
print(f'\nMFE Short (pips):')
print(df_labels['mfe_short_pips'].describe())
print(f'\nTrail stop used (pips):')
print(df_labels['trail_stop_pips'].describe())
print(f'\nLong trade duration (bars):')
print(df_labels['trail_long_bars'].describe())

# How often is MFE > 30 pips?
for threshold in [20, 30, 40, 50, 60]:
    n_long = (df_labels['mfe_long_pips'] > threshold).sum()
    n_short = (df_labels['mfe_short_pips'] > threshold).sum()
    total = df_labels['mfe_long_pips'].notna().sum()
    print(f'MFE > {threshold} pips — Long: {n_long} ({100*n_long/total:.1f}%) | Short: {n_short} ({100*n_short/total:.1f}%)')

del df_1h, df_labels

## 3. Additional Contextual Features

On top of the 64 microstructure features, we add:

| Category | Features | Rationale |
|----------|----------|-----------|
| **Feature momentum** | 3h/6h/12h deltas of key features | Detect regime *transitions*, not just states |
| **ATR context** | ATR_24, ATR ratio (short/long) | Volatility compression → breakout signal |
| **Session** | London/NY flags, hour-of-day | Big moves cluster at session opens |
| **Range position** | Price vs 24h/48h high-low | Breakouts from range extremes travel further |
| **Candle structure** | Body ratio, upper/lower wicks | Momentum candles vs dojis |

In [ ]:
def compute_extra_features(df_1h, df_micro, pair):
    """Compute additional features on top of existing microstructure features.
    
    Args:
        df_1h: 1H OHLCV (datetime index)
        df_micro: existing microstructure features (datetime index)
        pair: pair name
    
    Returns:
        DataFrame with extra feature columns
    """
    pip = PIP_SIZE[pair]
    feat = pd.DataFrame(index=df_micro.index)
    
    # Align 1H data to microstructure index
    df_1h = df_1h.reindex(df_micro.index)
    
    o = df_1h['open']
    h = df_1h['high']
    l = df_1h['low']
    c = df_1h['close']
    v = df_1h['volume']
    
    # ═══════════════════════════════════════════════════════════════
    # ATR CONTEXT
    # ═══════════════════════════════════════════════════════════════
    tr = np.maximum(h - l, np.maximum(np.abs(h - c.shift(1)), np.abs(l - c.shift(1))))
    feat['atr_6'] = tr.rolling(6, min_periods=3).mean() / pip
    feat['atr_24'] = tr.rolling(24, min_periods=6).mean() / pip
    feat['atr_72'] = tr.rolling(72, min_periods=24).mean() / pip
    
    # ATR compression ratio: short-term vs long-term
    # Low ratio = volatility squeeze → potential breakout
    feat['atr_ratio_6_24'] = feat['atr_6'] / feat['atr_24'].clip(lower=1e-10)
    feat['atr_ratio_6_72'] = feat['atr_6'] / feat['atr_72'].clip(lower=1e-10)
    
    # ═══════════════════════════════════════════════════════════════
    # RANGE POSITION
    # ═══════════════════════════════════════════════════════════════
    # Where is current price within recent range? 0 = at low, 1 = at high
    high_24 = h.rolling(24, min_periods=6).max()
    low_24 = l.rolling(24, min_periods=6).min()
    range_24 = high_24 - low_24
    feat['range_pos_24'] = (c - low_24) / range_24.clip(lower=1e-10)
    feat['range_width_24'] = range_24 / pip
    
    high_48 = h.rolling(48, min_periods=12).max()
    low_48 = l.rolling(48, min_periods=12).min()
    range_48 = high_48 - low_48
    feat['range_pos_48'] = (c - low_48) / range_48.clip(lower=1e-10)
    feat['range_width_48'] = range_48 / pip
    
    # Distance from 24h extremes (in ATR units — dimensionless)
    atr_price = tr.rolling(24, min_periods=6).mean()  # ATR in price units
    feat['dist_from_24h_high'] = (high_24 - c) / atr_price.clip(lower=1e-10)
    feat['dist_from_24h_low'] = (c - low_24) / atr_price.clip(lower=1e-10)
    
    # ═══════════════════════════════════════════════════════════════
    # SESSION FLAGS
    # ═══════════════════════════════════════════════════════════════
    hour = df_micro.index.hour
    feat['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    feat['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    feat['dow_sin'] = np.sin(2 * np.pi * df_micro.index.dayofweek / 5)
    feat['dow_cos'] = np.cos(2 * np.pi * df_micro.index.dayofweek / 5)
    
    # Session flags (UTC times)
    feat['is_london'] = ((hour >= 7) & (hour < 16)).astype(np.float32)
    feat['is_ny'] = ((hour >= 13) & (hour < 22)).astype(np.float32)
    feat['is_overlap'] = ((hour >= 13) & (hour < 16)).astype(np.float32)  # London+NY overlap
    feat['is_asia'] = ((hour >= 0) & (hour < 7)).astype(np.float32)
    
    # ═══════════════════════════════════════════════════════════════
    # CANDLE STRUCTURE
    # ═══════════════════════════════════════════════════════════════
    body = (c - o).abs()
    full_range = (h - l).clip(lower=1e-10)
    feat['body_ratio'] = body / full_range  # 1 = marubozu, 0 = doji
    feat['upper_wick_ratio'] = (h - np.maximum(o, c)) / full_range
    feat['lower_wick_ratio'] = (np.minimum(o, c) - l) / full_range
    feat['candle_direction'] = np.sign(c - o)  # +1 bullish, -1 bearish
    
    # Consecutive candle direction
    direction = feat['candle_direction']
    feat['consec_bullish'] = direction.rolling(6, min_periods=1).apply(
        lambda x: (x > 0).sum(), raw=True
    )
    feat['consec_bearish'] = direction.rolling(6, min_periods=1).apply(
        lambda x: (x < 0).sum(), raw=True
    )
    
    # ═══════════════════════════════════════════════════════════════
    # FEATURE MOMENTUM (deltas of key microstructure features)
    # ═══════════════════════════════════════════════════════════════
    key_micro_features = [
        'hurst_6h', 'vr_5', 'entropy_norm', 'order_imbalance',
        'kyle_lambda', 'jump_ratio', 'rv_yang_zhang'
    ]
    
    for col in key_micro_features:
        if col in df_micro.columns:
            series = df_micro[col]
            feat[f'{col}_delta_3h'] = series.diff(3)
            feat[f'{col}_delta_6h'] = series.diff(6)
            feat[f'{col}_delta_12h'] = series.diff(12)
    
    # ═══════════════════════════════════════════════════════════════
    # VOLUME CONTEXT
    # ═══════════════════════════════════════════════════════════════
    feat['volume_ratio_6'] = v / v.rolling(6, min_periods=2).mean().clip(lower=1)
    feat['volume_ratio_24'] = v / v.rolling(24, min_periods=6).mean().clip(lower=1)
    
    # Convert to float32
    for col in feat.columns:
        feat[col] = feat[col].astype(np.float32)
    
    return feat

print(f'Extra features function defined.')

In [ ]:
# Test on one pair
df_micro_test = load_microstructure_features('EURUSD')
df_1h_test = pd.read_parquet(PROCESSED_DIR / 'EURUSD_1H.parquet')
if 'datetime' in df_1h_test.columns:
    df_1h_test = df_1h_test.set_index('datetime')
df_1h_test.index = pd.to_datetime(df_1h_test.index)

df_extra = compute_extra_features(df_1h_test, df_micro_test, 'EURUSD')

print(f'Extra features: {df_extra.shape[1]} columns')
print(f'\nNew feature columns:')
for c in sorted(df_extra.columns):
    print(f'  {c}')

del df_micro_test, df_1h_test, df_extra

## 4. Build Full Dataset — All Pairs

For each pair:
1. Load microstructure features (64 cols)
2. Load 1H OHLCV
3. Compute MFE labels
4. Compute extra features
5. Merge and save to `features_6/`

In [ ]:
import gc

pair_stats = {}

for pair in ALL_PAIRS:
    print(f'\n{"=" * 50}')
    print(f'Processing {pair}')
    print(f'{"=" * 50}')
    
    # 1. Load microstructure features
    df_micro = load_microstructure_features(pair)
    micro_cols = [c for c in df_micro.columns if c != 'pair']
    print(f'  Microstructure: {len(micro_cols)} features, {len(df_micro)} rows')
    
    # 2. Load 1H OHLCV
    df_1h = pd.read_parquet(PROCESSED_DIR / f'{pair}_1H.parquet')
    if 'datetime' in df_1h.columns:
        df_1h = df_1h.set_index('datetime')
    df_1h.index = pd.to_datetime(df_1h.index)
    print(f'  1H OHLCV: {len(df_1h)} rows')
    
    # 3. Compute MFE labels
    df_labels = compute_mfe_labels(df_1h, pair)
    print(f'  MFE labels computed')
    
    # 4. Compute extra features
    df_extra = compute_extra_features(df_1h, df_micro, pair)
    print(f'  Extra features: {df_extra.shape[1]} columns')
    
    # 5. Merge everything
    # Start with microstructure features (drop 'pair' col, we'll add it back)
    df_out = df_micro.drop(columns=['pair'], errors='ignore').copy()
    
    # Add extra features
    df_out = df_out.join(df_extra, how='left')
    
    # Add MFE labels
    df_out = df_out.join(df_labels, how='left')
    
    # Add pair identifier
    df_out['pair'] = pair
    
    # Convert float64 to float32 where possible
    float64_cols = df_out.select_dtypes(include=[np.float64]).columns
    df_out[float64_cols] = df_out[float64_cols].astype(np.float32)
    
    # Stats
    n_total = df_out['mfe_long_pips'].notna().sum()
    n_big_long = (df_out['mfe_long_pips'] > 30).sum()
    n_big_short = (df_out['mfe_short_pips'] > 30).sum()
    label_cols = ['mfe_long_pips', 'mfe_short_pips', 'trail_long_bars', 'trail_short_bars', 'trail_stop_pips', 'mfe_atr_24']
    pair_stats[pair] = {
        'rows': len(df_out),
        'total_features': len([c for c in df_out.columns if c not in ['pair'] + label_cols]),
        'big_long_30': n_big_long,
        'big_short_30': n_big_short,
        'pct_big_long': 100 * n_big_long / max(n_total, 1),
        'pct_big_short': 100 * n_big_short / max(n_total, 1),
    }
    
    # Save
    out_path = OUTPUT_DIR / f'{pair}_features.parquet'
    df_out.to_parquet(out_path)
    print(f'  Saved: {out_path} ({df_out.shape[0]} rows × {df_out.shape[1]} cols)')
    print(f'  MFE>30 pips — Long: {n_big_long} ({pair_stats[pair]["pct_big_long"]:.1f}%) | Short: {n_big_short} ({pair_stats[pair]["pct_big_short"]:.1f}%)')
    
    del df_micro, df_1h, df_labels, df_extra, df_out
    gc.collect()

print(f'\n{"=" * 50}')
print('ALL PAIRS COMPLETE')
print(f'{"=" * 50}')

## 5. Summary

In [ ]:
# Summary table
print(f'{"Pair":<10} {"Rows":>8} {"Features":>10} {"Long>30p":>10} {"Short>30p":>10} {"Long%":>8} {"Short%":>8}')
print('-' * 70)
for pair, s in pair_stats.items():
    print(f'{pair:<10} {s["rows"]:>8,} {s["total_features"]:>10} {s["big_long_30"]:>10,} {s["big_short_30"]:>10,} {s["pct_big_long"]:>7.1f}% {s["pct_big_short"]:>7.1f}%')

# Total
total_rows = sum(s['rows'] for s in pair_stats.values())
total_long = sum(s['big_long_30'] for s in pair_stats.values())
total_short = sum(s['big_short_30'] for s in pair_stats.values())
print('-' * 70)
print(f'{"TOTAL":<10} {total_rows:>8,} {"":>10} {total_long:>10,} {total_short:>10,}')

# Output files
print(f'\nOutput directory: {OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.glob('*.parquet')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name}: {size_mb:.1f} MB')

In [ ]:
# Quick sanity check — load one file back and verify structure
df_check = pd.read_parquet(OUTPUT_DIR / 'EURUSD_features.parquet')

print(f'Shape: {df_check.shape}')
print(f'Date range: {df_check.index.min().date()} to {df_check.index.max().date()}')
print(f'\nColumn groups:')

micro_cols = [c for c in df_check.columns if c not in ['pair'] and not c.startswith(('mfe_', 'trail_', 'atr_', 'range_', 'hour_', 'dow_', 'is_', 'body_', 'upper_', 'lower_', 'candle_', 'consec_', 'volume_ratio', 'dist_from')) and '_delta_' not in c]
extra_cols = [c for c in df_check.columns if c.startswith(('atr_ratio', 'range_', 'hour_', 'dow_', 'is_', 'body_', 'upper_', 'lower_', 'candle_', 'consec_', 'volume_ratio', 'dist_from')) or '_delta_' in c]
label_cols = [c for c in df_check.columns if c.startswith(('mfe_', 'trail_stop', 'trail_long', 'trail_short'))]
atr_label = [c for c in df_check.columns if c == 'atr_24']

print(f'  Microstructure features: {len(micro_cols)}')
print(f'  Extra contextual features: {len(extra_cols)}')
print(f'  MFE label columns: {len(label_cols) + len(atr_label)}')
print(f'  Total: {len(df_check.columns)}')

print(f'\nMFE label stats:')
print(df_check[['mfe_long_pips', 'mfe_short_pips', 'trail_long_bars', 'trail_short_bars']].describe())

del df_check